# Channel EDA — Hit-Level Feature Distributions

Per-channel TP vs FP distributions for candidate ranking.  
Unit: (spectrum, candidate compound) pair.  
TP = correct candidate (hit_label=1), FP = wrong candidate (hit_label=0).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

ft = pd.read_csv('../data/feature_table_v2.csv')
tp = ft[ft['hit_label'] == 1]
fp = ft[ft['hit_label'] == 0]
print(f'TP: {len(tp):,}  FP: {len(fp):,}  Total: {len(ft):,}')

In [ ]:
def channel_plot(col, tp_vals, fp_vals, bins=80, xlim=None, log_scale=False, title=None):
    """Histogram overlay + summary stats + univariate AUC for one channel."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Left: histogram overlay
    ax = axes[0]
    kw = dict(bins=bins, density=True, alpha=0.5, edgecolor='none')
    if xlim:
        range_ = xlim
    else:
        combined = np.concatenate([tp_vals, fp_vals])
        q01, q99 = np.nanpercentile(combined, [1, 99])
        range_ = (q01, q99)
    
    ax.hist(tp_vals, label=f'TP (n={len(tp_vals):,})', color='steelblue', range=range_, **kw)
    ax.hist(fp_vals, label=f'FP (n={len(fp_vals):,})', color='salmon', range=range_, **kw)
    ax.set_xlabel(col)
    ax.set_ylabel('density')
    ax.set_title(title or col)
    ax.legend()
    if log_scale:
        ax.set_yscale('log')
    
    # Right: summary table
    ax2 = axes[1]
    ax2.axis('off')
    rows = []
    for name, vals in [('TP', tp_vals), ('FP', fp_vals)]:
        rows.append([name, f'{len(vals):,}', f'{np.nanmean(vals):.3f}',
                      f'{np.nanmedian(vals):.3f}', f'{np.nanstd(vals):.3f}',
                      f'{np.nanmin(vals):.3f}', f'{np.nanmax(vals):.3f}'])
    
    # Univariate AUC
    valid = ft[[col, 'hit_label']].dropna()
    if len(valid) > 0:
        auc = roc_auc_score(valid['hit_label'], valid[col])
        auc_str = f'{auc:.3f}'
    else:
        auc_str = 'N/A'
    
    table = ax2.table(cellText=rows,
                      colLabels=['', 'n', 'mean', 'median', 'std', 'min', 'max'],
                      loc='upper center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    ax2.set_title(f'Univariate AUC = {auc_str}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    return auc_str

## 1. Continuous Channels

In [ ]:
# entropy_similarity — primary spectral match channel
channel_plot('entropy_similarity',
             tp['entropy_similarity'].dropna().values,
             fp['entropy_similarity'].dropna().values,
             xlim=(0, 1), title='entropy_similarity (spectral match)')

In [ ]:
# delta_mda — mass accuracy (clipped to reasonable range)
# Outliers > 50 mDa are likely adduct parse failures or genuine annotation errors
tp_mda = tp['delta_mda'].dropna().values
fp_mda = fp['delta_mda'].dropna().values
channel_plot('delta_mda', tp_mda, fp_mda, xlim=(0, 20),
             title='delta_mda (mass accuracy, mDa) — clipped to 0–20')

# What fraction is > 50 mDa?
print(f'TP > 50 mDa: {(tp_mda > 50).sum()}/{len(tp_mda)} ({100*(tp_mda>50).mean():.1f}%)')
print(f'FP > 50 mDa: {(fp_mda > 50).sum()}/{len(fp_mda)} ({100*(fp_mda>50).mean():.1f}%)')

In [ ]:
# signed_delta_rt — RT prediction error (signed)
tp_rt = tp['signed_delta_rt'].dropna().values
fp_rt = fp['signed_delta_rt'].dropna().values
channel_plot('signed_delta_rt', tp_rt, fp_rt, xlim=(-100, 100),
             title='signed_delta_rt (observed − predicted RT, seconds)')

In [ ]:
# sim_gap — this candidate's similarity vs next best candidate
# Positive = you're the top hit, Negative = someone else is better
tp_gap = tp['sim_gap'].dropna().values
fp_gap = fp['sim_gap'].dropna().values
channel_plot('sim_gap', tp_gap, fp_gap, xlim=(-1, 1),
             title='sim_gap (this candidate sim − best other candidate sim)')

In [ ]:
# spectral_entropy — query spectrum complexity (context feature)
channel_plot('spectral_entropy',
             tp['spectral_entropy'].dropna().values,
             fp['spectral_entropy'].dropna().values,
             title='spectral_entropy (query spectrum entropy)')

## 2. Binary / Discrete Channels

In [ ]:
# Binary and discrete channel summary
binary_cols = ['hit_is_isf', 'hit_is_dubious', 'hit_isf_no_ok', 'compound_has_ok_adduct', 'polarity']
discrete_cols = ['n_candidate_adducts', 'n_candidates']

print('=== Binary channels: TP rate vs FP rate ===')
print(f'{"channel":30s}  {"TP rate":>8s}  {"FP rate":>8s}  {"logLR(1)":>8s}  {"logLR(0)":>8s}  {"AUC":>6s}')
print('-' * 90)
for col in binary_cols:
    tp_rate = tp[col].mean()
    fp_rate = fp[col].mean()
    # Laplace-smoothed logLR
    alpha = 1.0
    tp_p = (tp[col].sum() + alpha) / (len(tp) + 2 * alpha)
    fp_p = (fp[col].sum() + alpha) / (len(fp) + 2 * alpha)
    logLR_1 = np.log(tp_p / fp_p)
    logLR_0 = np.log((1 - tp_p) / (1 - fp_p))
    
    valid = ft[[col, 'hit_label']].dropna()
    auc = roc_auc_score(valid['hit_label'], valid[col])
    print(f'{col:30s}  {tp_rate:8.3f}  {fp_rate:8.3f}  {logLR_1:+8.3f}  {logLR_0:+8.3f}  {auc:.3f}')

print()
print('=== Discrete channels ===')
for col in discrete_cols:
    print(f'\n{col}:')
    for lbl, name in [(1, 'TP'), (0, 'FP')]:
        sub = ft[ft['hit_label']==lbl][col]
        print(f'  {name}: mean={sub.mean():.2f}  median={sub.median():.0f}  '
              f'mode={sub.mode().iloc[0]:.0f}  max={sub.max():.0f}')
    valid = ft[[col, 'hit_label']].dropna()
    auc = roc_auc_score(valid['hit_label'], valid[col])
    print(f'  AUC = {auc:.3f}')

## 3. Pairwise Correlations (Independence Check)

In [ ]:
# Spearman correlation among all channels, split by TP/FP
all_channels = ['entropy_similarity', 'delta_mda', 'signed_delta_rt', 'sim_gap',
                'spectral_entropy', 'hit_is_isf', 'hit_isf_no_ok',
                'compound_has_ok_adduct', 'n_candidate_adducts', 'polarity']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (subset, title) in zip(axes, [(tp, 'TP (correct candidates)'), (fp, 'FP (wrong candidates)')]):
    corr = subset[all_channels].corr(method='spearman')
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(all_channels)))
    ax.set_yticks(range(len(all_channels)))
    ax.set_xticklabels([c.replace('_', '\n') for c in all_channels], rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels([c.replace('_', '\n') for c in all_channels], fontsize=8)
    ax.set_title(title)
    # Annotate cells
    for i in range(len(all_channels)):
        for j in range(len(all_channels)):
            v = corr.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 0.5 else 'black')
fig.colorbar(im, ax=axes, shrink=0.8, label='Spearman ρ')
plt.suptitle('Channel Correlations — Independence Check', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Univariate AUC Summary

In [ ]:
# Compute AUC for all channels (directional: some channels are higher-is-better, some lower)
aucs = {}
for col in all_channels:
    valid = ft[[col, 'hit_label']].dropna()
    raw_auc = roc_auc_score(valid['hit_label'], valid[col])
    aucs[col] = max(raw_auc, 1 - raw_auc)  # best direction

# Sort by AUC descending
aucs_sorted = sorted(aucs.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
names = [a[0] for a in aucs_sorted]
vals = [a[1] for a in aucs_sorted]
bars = ax.barh(range(len(names)), vals, color='steelblue')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Univariate AUC (best direction)')
ax.set_title('Channel Discriminative Power — Univariate AUC')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
for i, v in enumerate(vals):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0.45, 1.0)
ax.invert_yaxis()
plt.tight_layout()
plt.show()